# 01 – Finding and downloading the data (NGC 3132, NIRCam)

**What:** Find the stage-3 images (`_i2d.fits`) of the six NIRCam filters from programme 2733 in MAST, check their size and download them.

**Why:** A JWST observation often has thousands of data products: raw data, intermediate stages, catalogues. We need exactly one file per filter, the fully combined image. Comparison: from a whole microscope folder (single tiles, z-stacks, metadata) you only take the stitched final image.

**Steps:**
1. Search observations (as in the setup check, now NIRCam only)
2. Get the product list and filter it to `_i2d.fits`
3. Check the size **before** downloading
4. Download the smallest file first (test run), then the rest

**Result:** 6 i2d files (filtered from 1956 products), all `PUBLIC`, together ≈2.49 GB, stored in `data/ngc3132_nircam/` (not in git). All downloads with status `COMPLETE`.

In [1]:
from pathlib import Path
from astroquery.mast import Observations

# Data live outside git (see .gitignore). Notebook runs in notebooks/, so go one level up.
DATA_DIR = Path("..") / "data" / "ngc3132_nircam"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Download folder:", DATA_DIR.resolve())

Download folder: /Users/inu/Projects/astro-pipeline/data/ngc3132_nircam


## 1. Search observations

`query_criteria` searches the **observation list**, i.e. *what* was measured – no files yet.
- `calib_level=3`: only the fully combined products, not the single exposures
- `instrument_name="NIRCAM/IMAGE"`: we saw this value in the setup check

Expectation: 6 rows, one per filter.

In [2]:
obs = Observations.query_criteria(
    obs_collection="JWST",
    proposal_id="2733",
    instrument_name="NIRCAM/IMAGE",
    calib_level=3,
)
print(len(obs), "NIRCam observations")
obs["obs_id", "filters", "t_exptime"]

6 NIRCam observations


obs_id,filters,t_exptime
str36,str11,float64
jw02733-o001_t001_nircam_clear-f187n,F187N,2319.144
jw02733-o001_t001_nircam_clear-f090w,F090W,1460.2000000000003
jw02733-o001_t001_nircam_clear-f356w,F356W,1460.2000000000003
jw02733-o001_t001_nircam_clear-f212n,F212N,2319.144
jw02733-o001_t001_nircam_f405n-f444w,F444W;F405N,2319.144
jw02733-o001_t001_nircam_f444w-f470n,F444W;F470N,2319.144


## 2. Get and filter the product list

`get_product_list` returns **all files** belonging to these observations, including the single exposures the final image was built from. That is why the list is long.

With `filter_products` we keep only:
- `productType="SCIENCE"`: image data, no auxiliary files
- `productSubGroupDescription="I2D"`: the combined image resampled onto a sky grid (*i*maging, *2D*)
- `calib_level=[3]`: stage 3 only

The column `dataRights` shows whether the data are public (`PUBLIC`). For an Early Release Observation they should be.

In [3]:
products = Observations.get_product_list(obs)
print(len(products), "products in total")

i2d = Observations.filter_products(
    products,
    productType="SCIENCE",
    productSubGroupDescription="I2D",
    calib_level=[3],
)
print(len(i2d), "i2d products after filtering")
i2d["productFilename", "size", "dataRights"]

1956 products in total
6 i2d products after filtering


productFilename,size,dataRights
str51,int64,str6
jw02733-o001_t001_nircam_clear-f090w_i2d.fits,673997760,PUBLIC
jw02733-o001_t001_nircam_clear-f356w_i2d.fits,155134080,PUBLIC
jw02733-o001_t001_nircam_f405n-f444w_i2d.fits,155134080,PUBLIC
jw02733-o001_t001_nircam_f444w-f470n_i2d.fits,155134080,PUBLIC
jw02733-o001_t001_nircam_clear-f187n_i2d.fits,674118720,PUBLIC
jw02733-o001_t001_nircam_clear-f212n_i2d.fits,674259840,PUBLIC


## 3. Check the size

`size` is given in bytes. We convert to MB/GB and sort in ascending order. If clearly more than 6 files or an unexpectedly large total show up here, do **not** continue. Then we first look at what is in the list.

In [4]:
i2d.sort("size")
for row in i2d:
    print(f"{row['productFilename']:<55} {row['size']/1e6:8.1f} MB")
print(f"\nTotal: {i2d['size'].sum()/1e9:.2f} GB")

jw02733-o001_t001_nircam_clear-f356w_i2d.fits              155.1 MB
jw02733-o001_t001_nircam_f405n-f444w_i2d.fits              155.1 MB
jw02733-o001_t001_nircam_f444w-f470n_i2d.fits              155.1 MB
jw02733-o001_t001_nircam_clear-f090w_i2d.fits              674.0 MB
jw02733-o001_t001_nircam_clear-f187n_i2d.fits              674.1 MB
jw02733-o001_t001_nircam_clear-f212n_i2d.fits              674.3 MB

Total: 2.49 GB


## 4a. Test run: download the smallest file

One file first, to check that the download works and where the file ends up – like a test image before starting the time series.

`flat=True` puts the file directly into `DATA_DIR`, without the nested MAST folder structure. The manifest shows the `Status` of each file (should be `COMPLETE`) and the local path.

In [5]:
manifest = Observations.download_products(i2d[:1], download_dir=str(DATA_DIR), flat=True)
manifest

Local Path,Status,Message,URL
str68,str8,object,object
../data/ngc3132_nircam/jw02733-o001_t001_nircam_clear-f356w_i2d.fits,COMPLETE,None,None


## 4b. Download the remaining files

Only run this if 4a worked: set the switch to `True`. astroquery normally skips files that are already present. I am not entirely sure whether this works with `flat=True` in every version – the message when running shows it.

In [6]:
DOWNLOAD_ALL = True   # set to True after the test download worked

if DOWNLOAD_ALL:
    manifest_all = Observations.download_products(i2d, download_dir=str(DATA_DIR), flat=True)
    display(manifest_all["Local Path", "Status"])
else:
    print("Skipped. Set DOWNLOAD_ALL = True to download all i2d files.")

INFO: Found cached file ../data/ngc3132_nircam/jw02733-o001_t001_nircam_clear-f356w_i2d.fits with expected size 155139840. [astroquery.query]


Local Path,Status
str68,str8
../data/ngc3132_nircam/jw02733-o001_t001_nircam_clear-f356w_i2d.fits,COMPLETE
../data/ngc3132_nircam/jw02733-o001_t001_nircam_f405n-f444w_i2d.fits,COMPLETE
../data/ngc3132_nircam/jw02733-o001_t001_nircam_f444w-f470n_i2d.fits,COMPLETE
../data/ngc3132_nircam/jw02733-o001_t001_nircam_clear-f090w_i2d.fits,COMPLETE
../data/ngc3132_nircam/jw02733-o001_t001_nircam_clear-f187n_i2d.fits,COMPLETE
../data/ngc3132_nircam/jw02733-o001_t001_nircam_clear-f212n_i2d.fits,COMPLETE


## 5. Check: what is stored locally now?

In [8]:
for f in sorted(DATA_DIR.glob("*_i2d.fits")):
    print(f"{f.name:<55} {f.stat().st_size/1e6:8.1f} MB")

jw02733-o001_t001_nircam_clear-f090w_i2d.fits              674.0 MB
jw02733-o001_t001_nircam_clear-f187n_i2d.fits              674.1 MB
jw02733-o001_t001_nircam_clear-f212n_i2d.fits              674.1 MB
jw02733-o001_t001_nircam_clear-f356w_i2d.fits              155.1 MB
jw02733-o001_t001_nircam_f405n-f444w_i2d.fits              155.1 MB
jw02733-o001_t001_nircam_f444w-f470n_i2d.fits              155.1 MB


## Conclusion

- The filter chain `SCIENCE` + `I2D` + `calib_level=[3]` reliably reduces 1956 products to exactly one file per filter.
- Short-wavelength images (F090W, F187N, F212N) are ≈674 MB, long-wavelength images (F356W, F405N, F470N) ≈155 MB. The ratio ≈4.3 fits the pixel sizes (0.031″ vs. 0.063″): half the pixel edge → ≈4× as many pixels. Important for choosing the reference grid in stage 3.
- Small oddity: the actual file sizes differ by a few kB from the `size` column of the product list (e.g. F212N: 674 121 600 instead of 674 259 840 bytes). astroquery itself expects exactly the local size in its cache check, and all downloads report `COMPLETE` – probably the catalogue entry is just slightly out of date (e.g. after reprocessing). Not certain; opening the files in stage 2 will show whether they are intact.
- The download skips files that are already present ("Found cached file"), also with `flat=True`.

**Next step:** stage 2 – open and understand an i2d file (extensions, header, WCS, histogram).